# Evaluación de la extracción de texto (Fase 2, conjunto held-out)

`calibrate_line_merge_tolerance.ipynb` usó los 100 documentos de `data/pdfs` + `data/labeled`
para **elegir** los parámetros de la heurística de reagrupamiento multilínea de
`LayoutExtractor` (`gap_fraction=0.25`, `min_tolerance=1.7` vertical; `min_tolerance=6.0`
horizontal). Reportar el resultado sobre ese mismo conjunto como "evaluación" sería
circular, porque es el conjunto que se usó para ajustar esos valores.

Este notebook mide el mismo tipo de error (¿el conjunto de bloques de texto extraídos
coincide con el JSON etiquetado manualmente?) pero sobre el conjunto **held-out** en
`evaluate/pdfs` + `evaluate/jsons`: 10 documentos que nunca se usaron para calibrar
tolerancias, ajustar la heurística de asociación ni entrenar el modelo. Los parámetros
usados son los que ya están fijados en el código — aquí no se vuelve a barrer ninguna
grilla, solo se corre una vez y se mide.

Se usa `LayoutExtractor.extract_from_doc`, el método real del pipeline completo, que
combina la extracción de texto digital (con la heurística de reagrupamiento multilínea)
y el OCR sobre las imágenes incrustadas, sin distinguir el origen del bloque al comparar
contra la verdad de campo.

Métrica principal: **ER (Error Rate) = bloques incorrectos / bloques totales**, tal como
se declaró en la Tabla 23 (Diseño experimental). Se reporta también la cantidad de
documentos con coincidencia exacta, con la misma definición que usa la calibración, para
poder comparar directamente el 90/100 obtenido ahí contra el resultado en datos nuevos.

In [1]:
import sys
import json
from pathlib import Path

import fitz
import pandas as pd

ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from extract.layout_extractor import LayoutExtractor

PDF_DIR = ROOT / "evaluate" / "pdfs"
LABELED_DIR = ROOT / "evaluate" / "jsons"

extractor = LayoutExtractor()

## 1. Conjunto held-out: PDFs con su JSON etiquetado correspondiente

In [2]:
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
labeled_by_stem = {}
for path in LABELED_DIR.glob("*.json"):
    labeled_by_stem[path.stem] = path

pairs = []
for pdf_path in pdf_paths:
    if pdf_path.stem in labeled_by_stem:
        pairs.append((pdf_path, labeled_by_stem[pdf_path.stem]))

print(f"Pares PDF + JSON etiquetado encontrados: {len(pairs)}")

Pares PDF + JSON etiquetado encontrados: 10


## 2. Extracción real vs. verdad de campo

`extract_actual_texts` reutiliza directamente `LayoutExtractor.extract_from_doc` (el
método real del pipeline completo, sin reimplementar nada), que ya combina el texto
digital reagrupado y el OCR de las imágenes incrustadas. `load_expected_texts` toma del
JSON etiquetado el texto de todas las entidades, sin filtrar por origen.

In [3]:
def load_expected_texts(json_path):
    with open(json_path, encoding="utf-8") as f:
        labeled_entities = json.load(f)
    return sorted(entity["text"] for entity in labeled_entities)


def extract_actual_texts(pdf_path):
    doc = fitz.open(pdf_path)
    elements, _images = extractor.extract_from_doc(doc)
    return sorted(element["text"] for element in elements)

## 3. Conteo de bloques correctos

Ambas listas se tratan como multiconjuntos: si un mismo texto aparece varias veces, cada
repetición cuenta por separado, sin sobrecontar.

In [4]:
def count_matched_blocks(expected_texts, actual_texts):
    remaining_actual = list(actual_texts)
    matched = 0
    for text in expected_texts:
        if text in remaining_actual:
            remaining_actual.remove(text)
            matched += 1
    return matched

## 4. Resultado por documento

In [5]:
rows = []
for pdf_path, json_path in pairs:
    expected_texts = load_expected_texts(json_path)
    actual_texts = extract_actual_texts(pdf_path)

    matched_blocks = count_matched_blocks(expected_texts, actual_texts)
    total_blocks = len(expected_texts)
    incorrect_blocks = total_blocks - matched_blocks
    error_rate = incorrect_blocks / total_blocks if total_blocks else 0.0

    rows.append({
        "documento": pdf_path.stem,
        "bloques_totales": total_blocks,
        "bloques_correctos": matched_blocks,
        "bloques_incorrectos": incorrect_blocks,
        "error_rate": error_rate,
        "coincidencia_exacta": actual_texts == expected_texts,
    })

results_df = pd.DataFrame(rows)
results_df

,documento,bloques_totales,bloques_correctos,bloques_incorrectos,error_rate,coincidencia_exacta
0,1106202601139174848500120080200000487350004873513,170,170,0,0.0,True
1,1706202601139174848500120080200000490960004909618,106,106,0,0.0,True
2,1706202601139174848500120080200000491290004912919,114,114,0,0.0,True
3,2206202601139175319500123020030003318020000308513,121,121,0,0.0,True
4,2906202601099001751400121650060010377100000000012,85,85,0,0.0,True
5,2908202601099000419600120372010001521100015211018,156,156,0,0.0,True
6,2908202601099151772300120020010002687650026876519,104,104,0,0.0,True
7,2908202601099151772300120020040001389570013895719,104,104,0,0.0,True
8,2908202601099151772300120020060001458990014589918,96,96,0,0.0,True
9,2908202601179098550400120250240000989170103303312,94,94,0,0.0,True


## 5. Resultado agregado (ER declarado en la Tabla 23)

In [6]:
total_blocks = results_df["bloques_totales"].sum()
total_correct = results_df["bloques_correctos"].sum()
total_incorrect = results_df["bloques_incorrectos"].sum()
overall_error_rate = total_incorrect / total_blocks
exact_match_docs = int(results_df["coincidencia_exacta"].sum())

print(f"Documentos evaluados: {len(results_df)}")
print(f"Documentos con coincidencia exacta: {exact_match_docs} de {len(results_df)}")
print(f"Bloques totales: {total_blocks}")
print(f"Bloques correctos: {total_correct}")
print(f"Bloques incorrectos: {total_incorrect}")
print(f"Error Rate (ER) global: {overall_error_rate:.4f} ({overall_error_rate:.2%})")

Documentos evaluados: 10
Documentos con coincidencia exacta: 10 de 10
Bloques totales: 1150
Bloques correctos: 1150
Bloques incorrectos: 0
Error Rate (ER) global: 0.0000 (0.00%)


### Conclusión

Sobre el conjunto held-out (10 documentos, nunca usados para calibrar ni entrenar), la
extracción de texto (digital + OCR) alcanza **ER=0.00%** (1150/1150 bloques correctos,
10/10 documentos con coincidencia exacta). El resultado no se degrada frente al 90/100
documentos y 99.0% de bloques obtenido durante la calibración en `data/` (sección 4.3.2)
— de hecho, en este conjunto en particular no aparecen ni los casos de sub-fusión
(`UN 26`, `THTHW0102`) documentados ahí como limitación conocida. Esto es consistente con
tratarse de un conjunto más pequeño (10 documentos vs. 100) y de la misma familia de
plantillas SRI, por lo que no contradice la limitación ya documentada — simplemente no la
observa en esta muestra.